## CTG-Net 
As CTG-Net is one of the most important model for this area, this notebook will be about exploring CTG-net and trying to apply the technique here

CTG-net is important in this area because it offers a quantitative, automated and less biased model for evaluating cardiatograms (CTGs).

This "Deep Neural Network-based classification of cardiotograms" claims to show an improved performance compared to conventional algorithms, such as Support Vector Machine (SVM) and k-means clustering as well as some other deep neural network models like Long Short-Term Memory (LSTM) models.

As a quantitative and automated diagnostic aid system, it enables early intervention for fetuses that are putatively abnormal, which can result in a reduction in the number of cases of hypoxic injury.

Biggest difference in CTG-net is, CTG-net can be trained with **high dimensional raw data** Unlike all the other models in this area, deep neural network models like CTG-Net did **NOT** require pre-processing. Furthermore, the DNN models tended to show slightly better higher performance when trained with the raw data compared to the denoised smoothed data. 

This notebook will further test and experiment CTG-net model technique. 

Source:  
[Deep neural network‐based
classification of cardiotocograms
outperformed conventional
algorithms](https://www.nature.com/articles/s41598-021-92805-9)





## Step 1: load the  Fetal Heart Rate (FHR) and Uterine Contradaction (UC) signal

Replicate the technique from CTG-Net: 

CTG-net uses raw data with some conditions: 

Classification: The 552 total samples were initially divided into 354 normal and 198 abnormal cases (according to pH and Apgar scores.)

Signal Cleaning: Repeated zero signals at the end of the samples were removed

Signal Extraction: The 30 minutes immediately preceding the last non-zero signals were extracted.

Selection: Only cases that satisfied the selection criteria (specifically, a signal loss less than 16%) were used for the final analysis, resulting in 52 normal and 26 abnormal cases.

Source:  
[Deep neural network‐based
classification of cardiotocograms
outperformed conventional
algorithms](https://www.nature.com/articles/s41598-021-92805-9)

In [ ]:
# necessary imports
import pandas as pd 
from pathlib import Path
import wfdb
import numpy as np
import tensorflow as tf
from tensorflow.keras.layers import (
    Input, Conv2D, DepthwiseConv2D, SeparableConv2D,
    BatchNormalization, Activation, AveragePooling2D,
    Dropout, Flatten, Dense
)
from tensorflow.keras.models import Model
from sklearn.preprocessing import LabelEncoder
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.optimizers import Adam
from sklearn.model_selection import KFold
from sklearn.metrics import (
    roc_auc_score,
    precision_score,
    recall_score,
    f1_score
)
from tensorflow.keras import backend as K
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from sklearn.utils import resample
from sklearn.model_selection import StratifiedKFold


In [35]:
# paths 
raw_dataset = Path("../data/raw_dataset")

records = [p.stem for p in raw_dataset.glob("*.hea")]
print(f"Found {len(records)} CTG records")



Found 552 CTG records


In [36]:
# Read one record to test
record = wfdb.rdrecord(raw_dataset / records[0])

signals = record.p_signal        
signal_names = record.sig_name   
fs = record.fs                  
print(f"Signal names: {signal_names}, Sampling rate: {fs} Hz")


Signal names: ['FHR', 'UC'], Sampling rate: 4 Hz


In [37]:
# Convert signals to a list
fhr = signals[:, 0].tolist()
uc = signals[:, 1].tolist()
print(f"First 10 FHR values: {fhr[:10]}")
print(f"First 10 UC values: {uc[:10]}")

First 10 FHR values: [150.5, 150.5, 151.0, 151.25, 151.25, 150.25, 150.25, 150.25, 148.75, 148.75]
First 10 UC values: [7.0, 8.5, 8.5, 7.5, 9.5, 8.5, 10.5, 12.0, 11.0, 11.5]


In [38]:
'''
Step 1: Classification 
CTG-net uses umblical arthery pH and Apgar scores:
Deliveries were classified as abnormal if the umbilical artery pH was lower than 7.20 or the Apgar score at 1 minute was lower than 7.
However, in my technique I use majority voting based on 9 obstetrician's assessment - as discussed before.
'''
# path to majority voting labels
Labels = '..\ExpertAnnotations\CTG_Majority_Vote_Labels_FINAL.csv'

# print number of normal and abnormal deliveries
labels_df = pd.read_csv(Labels)
num_normal = sum(labels_df['Clinical_Label'] == 'No Hypoxia (Normal)')
num_abnormal = sum(labels_df['Clinical_Label'] == 'Mild Hypoxia (Suspicious)')
print(f'Number of normal deliveries: {num_normal}')
print(f'Number of abnormal deliveries: {num_abnormal}')


Number of normal deliveries: 426
Number of abnormal deliveries: 126


In [39]:
'''
Step 2: Signal cleaning 
Repeated zero signals at the end of the samples were removed
'''
def remove_trailing_zeros(signal):
    """Remove trailing zeros from a signal."""
    if not isinstance(signal, list):
        raise ValueError("Input signal must be a list.")
    
    # Find the index of the last non-zero element
    last_non_zero_index = len(signal) - 1
    while last_non_zero_index >= 0 and signal[last_non_zero_index] == 0:
        last_non_zero_index -= 1
    
    # Return the signal up to the last non-zero element
    return signal[:last_non_zero_index + 1]

In [40]:
'''
Step 3: Signal Extraction
The 30 minutes immediately preceding the last non-zero signals were extracted.
'''
def extract_last_30_minutes(signal, sampling_rate=4):
    ''' reject short recordings and extract last 30 minutes of signal '''
    num_samples_30_minutes = 30 * 60 * sampling_rate

    if len(signal) < num_samples_30_minutes:
        return []  # reject short recordings

    return signal[-num_samples_30_minutes:]


In [41]:
'''
Step 4: Downsample signals to 1 Hz
The signals were originally sampled at 4 Hz. They were downsampled to 1 Hz by taking every fourth sample.
Paper explicitly states: “Signals were downsampled to 1 Hz for 30 minutes (1800 points)”
'''
def downsample_to_1hz(signal, original_fs=4, target_fs=1):
    factor = original_fs // target_fs
    return signal[::factor]


In [42]:
'''
Step 5: Selection
Only cases that satisfied the selection criteria (specifically, a signal loss less than 16%) were used for the final analysis
'''
def is_signal_acceptable(signal, threshold=0.16):
    """Check if the signal loss is within the acceptable threshold."""
    
    # Reject empty or invalid signals
    if not signal or len(signal) == 0:
        return False
    
    total_length = len(signal)
    zero_count = signal.count(0)
    signal_loss = zero_count / total_length
    
    return signal_loss < threshold


In [43]:
''' 
Step 6: Perform the above steps on the dataset 
'''
processed_records = []

for rec in records:
    record = wfdb.rdrecord(raw_dataset / rec)
    signals = record.p_signal

    fhr = signals[:, 0].tolist()
    uc  = signals[:, 1].tolist()

    # Cleaning 
    fhr = remove_trailing_zeros(fhr)
    uc  = remove_trailing_zeros(uc)

    # Extraction 
    fhr = extract_last_30_minutes(fhr, sampling_rate=record.fs)
    uc  = extract_last_30_minutes(uc, sampling_rate=record.fs)

    # Downsampling
    fhr = downsample_to_1hz(fhr, original_fs=fs, target_fs=1)
    uc  = downsample_to_1hz(uc, original_fs=fs, target_fs=1)

    # Selection
    if is_signal_acceptable(fhr) and is_signal_acceptable(uc):
        processed_records.append({
            "rec_id": rec,
            "FHR": fhr,
            "UC": uc
        })


In [44]:
data_df = pd.DataFrame(processed_records)
print(data_df.shape)


(220, 3)


In [45]:
labels_df = pd.read_csv(Labels)
print(labels_df.columns)
data_df['rec_id'] = data_df['rec_id'].astype(str)
labels_df['rec_id'] = labels_df['rec_id'].astype(str)
merged_df = data_df.merge(
    labels_df,
    left_on="rec_id",
    right_on="rec_id",
    how="inner"
)

print(merged_df.shape)
label_counts = merged_df['Clinical_Label'].value_counts()
print(label_counts)




Index(['rec_id', 'Majority_Vote_Label', 'Clinical_Label'], dtype='object')
(220, 5)
No Hypoxia (Normal)          164
Mild Hypoxia (Suspicious)     56
Name: Clinical_Label, dtype: int64


In [46]:
def normalize_signals(X):
    """
    Normalize FHR and UC signals independently per sample
    Paper likely used z-score normalization
    """
    X_norm = np.zeros_like(X, dtype=np.float32)
    
    for i in range(len(X)):
        # Normalize FHR (channel 0)
        fhr = X[i, 0, :, 0]
        fhr_mean = np.mean(fhr)
        fhr_std = np.std(fhr)
        if fhr_std > 0:
            X_norm[i, 0, :, 0] = (fhr - fhr_mean) / fhr_std
        else:
            X_norm[i, 0, :, 0] = fhr - fhr_mean
        
        # Normalize UC (channel 1)
        uc = X[i, 1, :, 0]
        uc_mean = np.mean(uc)
        uc_std = np.std(uc)
        if uc_std > 0:
            X_norm[i, 1, :, 0] = (uc - uc_mean) / uc_std
        else:
            X_norm[i, 1, :, 0] = uc - uc_mean
    
    return X_norm

X_normalized = normalize_signals(X)

## Step 2: Building CTG-net
**3 CNN Layers** : that extract temporal patterns and interrelationship between FHR and UC.

### Architecture:
Input Layer (2, 1800, 1)

Time direction conv. (2, 1800, 4)

Batch Normalisation (2, 1800, 4)

Depthwise 2D conv (1, 1800, 8)

Batch Normalisation (1, 1800, 8)

Activation (1, 1800, 8)

Average 2D Pooling (1, 450, 8)

Dropout (1, 450, 8)

Separable 2D Conv. (1, 450, 8)

Batch Normalisation (1, 450, 8)

Activation (1, 450, 8)

Average 2D Pooling (1,112,8)

Dropout (1, 112, 8)

Flatten(896)

Dense (2)

Sigmoid (2)

Total number of parameters: 2.130

In [47]:
def build_ctg_net(input_shape=(2, 1800, 1), dropout_rate=0.25):
    """
    CTG-Net architecture as per the paper
    - Configurable dropout rate
    - ~1,990 parameters
    """
    inputs = Input(shape=input_shape)

    # Time-direction convolution
    x = Conv2D(
        filters=4,
        kernel_size=(1, 3),
        padding="same",
        use_bias=False
    )(inputs)
    x = BatchNormalization()(x)

    # Depthwise convolution (captures FHR-UC interactions)
    x = DepthwiseConv2D(
        kernel_size=(2, 1),
        depth_multiplier=2,
        use_bias=False
    )(x)
    x = BatchNormalization()(x)
    x = Activation("relu")(x)

    # Average pooling + Dropout
    x = AveragePooling2D(pool_size=(1, 4))(x)
    x = Dropout(dropout_rate)(x)

    # Separable convolution
    x = SeparableConv2D(
        filters=8,
        kernel_size=(1, 3),
        padding="same",
        use_bias=False
    )(x)
    x = BatchNormalization()(x)
    x = Activation("relu")(x)

    # Average pooling + Dropout
    x = AveragePooling2D(pool_size=(1, 4))(x)
    x = Dropout(dropout_rate)(x)

    # Classification head
    x = Flatten()(x)
    outputs = Dense(2, activation="sigmoid")(x)

    model = Model(inputs, outputs, name="CTG-Net")
    return model

In [48]:
model = build_ctg_net()
model.summary()

Model: "CTG-Net"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_1 (InputLayer)      │ (None, 2, 1800, 1)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 2, 1800, 4)     │            12 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_3           │ (None, 2, 1800, 4)     │            16 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ depthwise_conv2d_1              │ (None, 1, 1800, 8)     │            16 │
│ (DepthwiseConv2D)               │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_4           │ (None, 1, 1800, 8)     │            32 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_2 (Activation)       │ (None, 1, 1800, 8)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ average_pooling2d_2             │ (None, 1, 450, 8)      │             0 │
│ (AveragePooling2D)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 1, 450, 8)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ separable_conv2d_1              │ (None, 1, 450, 8)      │            88 │
│ (SeparableConv2D)               │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_5           │ (None, 1, 450, 8)      │            32 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_3 (Activation)       │ (None, 1, 450, 8)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ average_pooling2d_3             │ (None, 1, 112, 8)      │             0 │
│ (AveragePooling2D)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 1, 112, 8)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_1 (Flatten)             │ (None, 896)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 2)              │         1,794 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,990 (7.77 KB)

 Trainable params: 1,950 (7.62 KB)

 Non-trainable params: 40 (160.00 B)

## Model Training: 

Input Data: The CTG-net took Fetal Heart Rate (FHR) and Uterine Contraction (UC) signals of 1800 time points (downsampled at 1 Hz for 30 minutes) as input.

Training Framework: The models were trained in TensorFlow using the Keras API on an Intel Core i7-8650U CPU (In the actual paper).

**Parameters:**

**Optimizer**: Adam optimizer.

**Loss Function**: Categorical cross-entropy.

**Overfitting Prevention**: A dropout rate of 25% was set in the CTG-net to avoid overfitting.

**Validation**: The models were tested using ten-fold cross-validation with both raw and smoothed CTG data.

According to the paper and the reported CTG-net architecture, it is highly likely that the class labels were represented using one-hot encoding. The network employs a final dense layer with two output units followed by a sigmoid activation, which is consistent with a two-dimensional probability vector corresponding to the normal and abnormal classes. Furthermore, the use of categorical cross-entropy as the loss function conventionally requires one-hot encoded target labels. Together, these architectural and training choices strongly suggest that binary class labels were encoded in one-hot form.




In [49]:
''' 
Step 1: Change labels from binary (normal, abnormal) to one- hot encoding
'''

# Label encoding
X = np.array([
    np.stack([row["FHR"], row["UC"]], axis=0)
    for _, row in merged_df.iterrows()
])[..., np.newaxis]

# Labels → one-hot encoding
le = LabelEncoder()
y_int = le.fit_transform(merged_df["Clinical_Label"])  # 0=Normal, 1=Abnormal
y = to_categorical(y_int, num_classes=2)



In [50]:
def find_best_f1_threshold(y_true, y_prob):
    """
    Finds the threshold in [0,1] that maximizes F1 score.
    Evaluates all meaningful thresholds (unique predicted probabilities).
    """
    best_f1 = -1
    best_thr = 0.5
    best_prec = 0.0
    best_rec = 0.0

    thresholds = np.unique(y_prob)  

    for thr in thresholds:
        y_pred = (y_prob >= thr).astype(int)

        if y_pred.sum() == 0:
            continue  
        prec = precision_score(y_true, y_pred, zero_division=0)
        rec = recall_score(y_true, y_pred, zero_division=0)
        f1 = f1_score(y_true, y_pred, zero_division=0)

        if f1 > best_f1:
            best_f1 = f1
            best_thr = thr
            best_prec = prec
            best_rec = rec

    return best_thr, best_f1, best_prec, best_rec

In [51]:
''' 
Prepare 10-fold cross-validation 
'''

kf = KFold(n_splits=10, shuffle=True, random_state=42)

# Separate normal and abnormal indices
normal_indices = np.where(y_int == 0)[0]
abnormal_indices = np.where(y_int == 1)[0]

n_iterations = 10
test_ratio = 0.1  # 9:1 split

auc_scores = []
f1_scores = []
precision_scores = []
recall_scores = []
thresholds_list = []

In [52]:
''' 
Configure Early Stopping for model training
'''
early_stop = EarlyStopping(
    monitor='loss',  
    patience=15,
    restore_best_weights=True
)


In [57]:
'''
Perform 10-fold CV
'''

# Outer loop: 10 independent experiments with different normal samples
for iteration in range(1, n_iterations + 1):
    print(f"Iteration {iteration}/{n_iterations}")
    
    # Step 1: Use ALL normal cases
    sampled_normal_indices = normal_indices.copy()
    
    # Step 2: Sample abnormal cases to match normal count (without replacement)
    np.random.seed(42 + iteration)
    sampled_abnormal_indices = resample(
        abnormal_indices,
        n_samples=len(normal_indices),  # Match normal count
        replace=False,
        random_state=42 + iteration
    )
    
    # Step 3: Combine into balanced dataset
    balanced_indices = np.concatenate([sampled_normal_indices, sampled_abnormal_indices])
    X_balanced = X_normalized[balanced_indices]
    y_balanced = y[balanced_indices]
    y_labels_balanced = y_balanced[:, 1]  # For stratification
    
    # Step 4: Apply 10-fold CV on this balanced dataset
    skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=42 + iteration)
    
    for fold, (train_idx, test_idx) in enumerate(skf.split(X_balanced, y_labels_balanced), 1):
        print(f"\n--- Fold {fold}/10 ---")
        
        X_train = X_balanced[train_idx]
        y_train = y_balanced[train_idx]
        X_test = X_balanced[test_idx]
        y_test = y_balanced[test_idx]
        
        # Print split info
        n_normal_train = (y_train[:, 0] == 1).sum()
        n_abnormal_train = (y_train[:, 1] == 1).sum()
        n_normal_test = (y_test[:, 0] == 1).sum()
        n_abnormal_test = (y_test[:, 1] == 1).sum()
        
        print(f"Train: Normal={n_normal_train}, Abnormal={n_abnormal_train}, Total={len(train_idx)}")
        print(f"Test:  Normal={n_normal_test}, Abnormal={n_abnormal_test}, Total={len(test_idx)}")
        
        # Build and train model (same as your code)
        K.clear_session()
        model = build_ctg_net()
        
        model.compile(
            optimizer=Adam(learning_rate=0.001),
            loss="categorical_crossentropy"
        )
        
        # Class weights (should be ~1.0 for both since balanced)
        class_weights = {
            0: 1.0,
            1: n_normal_train / n_abnormal_train
        }
        print(f"Class weights: 0={class_weights[0]:.2f}, 1={class_weights[1]:.2f}")
        
        # Callbacks
        reduce_lr = ReduceLROnPlateau(
            monitor='loss',
            factor=0.5,
            patience=15,
            min_lr=1e-7,
            verbose=0
        )
        
        early_stop = EarlyStopping(
            monitor='loss',
            patience=25,
            restore_best_weights=True,
            verbose=0
        )
        
        # Train
        print("Training...", end=" ", flush=True)
        history = model.fit(
            X_train, y_train,
            epochs=300,
            batch_size= 16,
            class_weight=class_weights,
            callbacks=[early_stop, reduce_lr],
            verbose=0
        )
        print(f"Done! ({len(history.history['loss'])} epochs, final loss={history.history['loss'][-1]:.4f})")
        
        # Predict and evaluate
        y_prob = model.predict(X_test, verbose=0)[:, 1]
        y_test_int = y_test[:, 1]
        
        print(f"Predictions: min={y_prob.min():.3f}, max={y_prob.max():.3f}, mean={y_prob.mean():.3f}, std={y_prob.std():.3f}")
        
        try:
            auc = roc_auc_score(y_test_int, y_prob)
            best_thr, best_f1, best_prec, best_rec = find_best_f1_threshold(y_test_int, y_prob)
            
            auc_scores.append(auc)
            f1_scores.append(best_f1)
            precision_scores.append(best_prec)
            recall_scores.append(best_rec)
            thresholds_list.append(best_thr)
            
            print(f"→ AUC: {auc:.3f} | F1: {best_f1:.3f} | Prec: {best_prec:.3f} | Rec: {best_rec:.3f} | Thr: {best_thr:.3f}")
            
        except Exception as e:
            print(f"✗ Error: {e}")

Iteration 1/10

--- Fold 1/10 ---
Train: Normal=50, Abnormal=50, Total=100
Test:  Normal=6, Abnormal=6, Total=12
Class weights: 0=1.00, 1=1.00
Training... Done! (136 epochs, final loss=0.0555)
Predictions: min=0.053, max=0.666, mean=0.298, std=0.202
→ AUC: 0.389 | F1: 0.667 | Prec: 0.500 | Rec: 1.000 | Thr: 0.053

--- Fold 2/10 ---
Train: Normal=50, Abnormal=50, Total=100
Test:  Normal=6, Abnormal=6, Total=12
Class weights: 0=1.00, 1=1.00
Training... Done! (182 epochs, final loss=0.0228)
Predictions: min=0.207, max=0.986, mean=0.679, std=0.246
→ AUC: 0.583 | F1: 0.706 | Prec: 0.545 | Rec: 1.000 | Thr: 0.254

--- Fold 3/10 ---
Train: Normal=50, Abnormal=51, Total=101
Test:  Normal=6, Abnormal=5, Total=11
Class weights: 0=1.00, 1=0.98
Training... Done! (140 epochs, final loss=0.0484)
Predictions: min=0.070, max=0.965, mean=0.495, std=0.310
→ AUC: 0.867 | F1: 0.833 | Prec: 0.714 | Rec: 1.000 | Thr: 0.431

--- Fold 4/10 ---
Train: Normal=50, Abnormal=51, Total=101
Test:  Normal=6, Abnormal

In [58]:
''' 
Reporting that matches with the original CTG-Net paper 
'''
print("\n===== CTG-Net 10-Fold CV Results =====")
print(f"Mean AUC:       {np.mean(auc_scores):.3f} ± {np.std(auc_scores):.3f}")
print(f"Mean F1:        {np.mean(f1_scores):.3f} ± {np.std(f1_scores):.3f}")
print(f"Mean Precision: {np.mean(precision_scores):.3f}")
print(f"Mean Recall:    {np.mean(recall_scores):.3f}")
#print(f"Mean Threshold: {np.mean(thresholds):.3f}")



===== CTG-Net 10-Fold CV Results =====
Mean AUC:       0.630 ± 0.170
Mean F1:        0.754 ± 0.078
Mean Precision: 0.657
Mean Recall:    0.935


## Paper Results
**Area Under the Curve (AUC):** The overall performance for various thresholds was evaluated using the AUC:

CTG-net (raw data): $0.73 \pm 0.04$

**F1 score** - CTG-Net: $0.67 \pm 0.03$

In [32]:
from sklearn.model_selection import StratifiedKFold

# Outer loop: 10 independent experiments with different normal samples
for iteration in range(1, n_iterations + 1):
    print(f"\n{'='*60}")
    print(f"Iteration {iteration}/{n_iterations}")
    print(f"{'='*60}")
    
    # Step 1: Sample 162 normal cases 
    np.random.seed(42 + iteration)
    sampled_normal_indices = resample(
        normal_indices,
        n_samples=len(abnormal_indices),  # Match abnormal count 
        replace=False,
        random_state=42 + iteration
    )
    
    # Step 2: Use ALL abnormal cases
    sampled_abnormal_indices = abnormal_indices.copy()
    
    # Step 3: Combine into balanced dataset
    balanced_indices = np.concatenate([sampled_normal_indices, sampled_abnormal_indices])
    X_balanced = X_normalized[balanced_indices]
    y_balanced = y[balanced_indices]
    y_labels_balanced = y_balanced[:, 1] 

    # Step 4: Apply 10-fold CV on this balanced dataset
    skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=42 + iteration)
    
    for fold, (train_idx, test_idx) in enumerate(skf.split(X_balanced, y_labels_balanced), 1):
        print(f"\n--- Fold {fold}/10 ---")
        
        X_train = X_balanced[train_idx]
        y_train = y_balanced[train_idx]
        X_test = X_balanced[test_idx]
        y_test = y_balanced[test_idx]
        
        # Print split info
        n_normal_train = (y_train[:, 0] == 1).sum()
        n_abnormal_train = (y_train[:, 1] == 1).sum()
        n_normal_test = (y_test[:, 0] == 1).sum()
        n_abnormal_test = (y_test[:, 1] == 1).sum()
        
        print(f"Train: Normal={n_normal_train}, Abnormal={n_abnormal_train}, Total={len(train_idx)}")
        print(f"Test:  Normal={n_normal_test}, Abnormal={n_abnormal_test}, Total={len(test_idx)}")
        
        # Build and train model
        K.clear_session()
        model = build_ctg_net()
        
        model.compile(
            optimizer=Adam(learning_rate=0.001),
            loss="categorical_crossentropy"
        )
        
        # Class weights (should be ~1.0 for both since balanced)
        class_weights = {
            0: 1.0,
            1: n_normal_train / n_abnormal_train
        }
        print(f"Class weights: 0={class_weights[0]:.2f}, 1={class_weights[1]:.2f}")
        
        # Callbacks
        reduce_lr = ReduceLROnPlateau(
            monitor='loss',
            factor=0.5,
            patience=15,
            min_lr=1e-7,
            verbose=0
        )
        
        early_stop = EarlyStopping(
            monitor='loss',
            patience=25,
            restore_best_weights=True,
            verbose=0
        )
        
        # Train
        print("Training...", end=" ", flush=True)
        history = model.fit(
            X_train, y_train,
            epochs=300,
            batch_size=8,
            class_weight=class_weights,
            callbacks=[early_stop, reduce_lr],
            verbose=0
        )
        print(f"Done! ({len(history.history['loss'])} epochs, final loss={history.history['loss'][-1]:.4f})")
        
        # Predict and evaluate
        y_prob = model.predict(X_test, verbose=0)[:, 1]
        y_test_int = y_test[:, 1]
        
        print(f"Predictions: min={y_prob.min():.3f}, max={y_prob.max():.3f}, mean={y_prob.mean():.3f}, std={y_prob.std():.3f}")
        
        try:
            auc = roc_auc_score(y_test_int, y_prob)
            best_thr, best_f1, best_prec, best_rec = find_best_f1_threshold(y_test_int, y_prob)
            
            auc_scores.append(auc)
            f1_scores.append(best_f1)
            precision_scores.append(best_prec)
            recall_scores.append(best_rec)
            thresholds_list.append(best_thr)
            
            print(f"→ AUC: {auc:.3f} | F1: {best_f1:.3f} | Prec: {best_prec:.3f} | Rec: {best_rec:.3f} | Thr: {best_thr:.3f}")
            
        except Exception as e:
            print(f"✗ Error: {e}")


Iteration 1/10

--- Fold 1/10 ---
Train: Normal=50, Abnormal=50, Total=100
Test:  Normal=6, Abnormal=6, Total=12
Class weights: 0=1.00, 1=1.00
Training... Done! (129 epochs, final loss=0.0334)
Predictions: min=0.069, max=0.943, mean=0.676, std=0.330
→ AUC: 0.444 | F1: 0.706 | Prec: 0.545 | Rec: 1.000 | Thr: 0.124

--- Fold 2/10 ---
Train: Normal=50, Abnormal=50, Total=100
Test:  Normal=6, Abnormal=6, Total=12
Class weights: 0=1.00, 1=1.00
Training... Done! (208 epochs, final loss=0.0117)
Predictions: min=0.014, max=0.998, mean=0.585, std=0.298
→ AUC: 0.583 | F1: 0.714 | Prec: 0.625 | Rec: 0.833 | Thr: 0.544

--- Fold 3/10 ---
Train: Normal=50, Abnormal=51, Total=101
Test:  Normal=6, Abnormal=5, Total=11
Class weights: 0=1.00, 1=0.98
Training... Done! (134 epochs, final loss=0.0272)
Predictions: min=0.005, max=0.899, mean=0.336, std=0.329
→ AUC: 0.733 | F1: 0.727 | Prec: 0.667 | Rec: 0.800 | Thr: 0.158

--- Fold 4/10 ---
Train: Normal=50, Abnormal=51, Total=101
Test:  Normal=6, Abnorma

In [56]:
''' 
Reporting that matches with the original CTG-Net paper 
'''
print("\n===== CTG-Net 10-Fold CV Results =====")
print(f"Mean AUC:       {np.mean(auc_scores):.3f} ± {np.std(auc_scores):.3f}")
print(f"Mean F1:        {np.mean(f1_scores):.3f} ± {np.std(f1_scores):.3f}")
print(f"Mean Precision: {np.mean(precision_scores):.3f}")
print(f"Mean Recall:    {np.mean(recall_scores):.3f}")
#print(f"Mean Threshold: {np.mean(thresholds):.3f}")


===== CTG-Net 10-Fold CV Results =====
Mean AUC:       0.652 ± 0.166
Mean F1:        0.760 ± 0.081
Mean Precision: 0.670
Mean Recall:    0.929


In [ ]:
# need access to the FULL pool of normal cases
# normal_indices_pool: array of ALL ~1,954 normal case indices
# abnormal_indices: array of ALL 162 abnormal case indices

for iteration in range(1, 11):  # 10 experiments
    print(f"\n{'='*60}")
    print(f"Experiment {iteration}/10")
    print(f"{'='*60}")
    
    # Step 1: Sample 162 normal cases from the FULL pool of ~1,954
    np.random.seed(42 + iteration)
    sampled_normal_indices = resample(
        processed_records,  # The FULL pool of normal cases
        n_samples=162,  # Fixed: sample exactly 162
        replace=False,
        random_state=42 + iteration
    )
    
    # Step 2: Use ALL 162 abnormal cases
    sampled_abnormal_indices = abnormal_indices.copy()  # All 162
    
    # Step 3: Create balanced dataset for this experiment
    experiment_indices = np.concatenate([sampled_normal_indices, sampled_abnormal_indices])
    X_experiment = X_normalized[experiment_indices]
    y_experiment = y[experiment_indices]
    y_labels_experiment = y_experiment[:, 1]
    
    print(f"Sampled dataset: {len(sampled_normal_indices)} normal + {len(sampled_abnormal_indices)} abnormal = {len(experiment_indices)} total")
    
    # Step 4: Apply 10-fold CV on THIS experiment's balanced dataset
    skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=42 + iteration)
    
    for fold, (train_idx, test_idx) in enumerate(skf.split(X_experiment, y_labels_experiment), 1):
        print(f"\n  Fold {fold}/10")
        
        X_train = X_experiment[train_idx]
        y_train = y_experiment[train_idx]
        X_test = X_experiment[test_idx]
        y_test = y_experiment[test_idx]
        
        # Print split info
        n_normal_train = (y_train[:, 0] == 1).sum()
        n_abnormal_train = (y_train[:, 1] == 1).sum()
        n_normal_test = (y_test[:, 0] == 1).sum()
        n_abnormal_test = (y_test[:, 1] == 1).sum()
        
        print(f"  Train: Normal={n_normal_train}, Abnormal={n_abnormal_train}, Total={len(train_idx)}")
        print(f"  Test:  Normal={n_normal_test}, Abnormal={n_abnormal_test}, Total={len(test_idx)}")
        
        # Build model
        K.clear_session()
        model = build_ctg_net()
        
        model.compile(
            optimizer=Adam(learning_rate=0.001),
            loss="categorical_crossentropy"
        )
        
        # Class weights (should be ~1.0 since balanced)
        # NOTE: Paper doesn't mention using class weights
        # You might want to remove this entirely
        class_weights = {
            0: 1.0,
            1: n_normal_train / n_abnormal_train
        }
        
        # Train
        print("  Training...", end=" ", flush=True)
        history = model.fit(
            X_train, y_train,
            epochs=300,
            batch_size=8,
            class_weight=class_weights,  # Consider removing if paper doesn't use
            callbacks=[early_stop, reduce_lr],
            verbose=0
        )
        print(f"Done! ({len(history.history['loss'])} epochs)")
        
        # Evaluate
        y_prob = model.predict(X_test, verbose=0)[:, 1]
        y_test_int = y_test[:, 1]
        
        auc = roc_auc_score(y_test_int, y_prob)
        best_thr, best_f1, best_prec, best_rec = find_best_f1_threshold(y_test_int, y_prob)
        
        auc_scores.append(auc)
        f1_scores.append(best_f1)
        # ... store other metrics
        
        print(f"  → AUC: {auc:.3f} | F1: {best_f1:.3f}")

In [59]:
print(processed_records)

[{'rec_id': '1005', 'FHR': [127.25, 128.75, 124.5, 121.5, 123.75, 125.25, 128.0, 129.5, 129.0, 129.75, 129.25, 131.0, 131.5, 134.25, 136.5, 140.25, 140.75, 114.75, 114.0, 114.5, 114.25, 114.5, 109.5, 103.25, 103.75, 100.5, 102.0, 97.75, 90.0, 92.75, 93.5, 92.25, 92.25, 92.25, 90.0, 90.0, 90.0, 0.0, 81.25, 86.75, 79.25, 80.75, 87.0, 87.0, 87.0, 0.0, 0.0, 0.0, 0.0, 100.5, 100.25, 101.5, 102.0, 103.0, 104.25, 109.75, 110.25, 110.5, 112.25, 116.0, 115.0, 113.0, 111.75, 111.25, 110.25, 112.0, 113.75, 114.5, 115.75, 114.75, 115.25, 115.0, 115.25, 117.75, 117.25, 116.5, 117.0, 114.25, 113.25, 113.5, 115.0, 116.5, 117.5, 118.5, 118.5, 115.25, 113.5, 113.25, 114.5, 114.0, 113.75, 114.0, 113.75, 113.75, 115.25, 115.5, 115.0, 114.5, 114.25, 113.5, 113.5, 113.0, 115.0, 115.0, 114.25, 115.5, 115.0, 117.5, 116.25, 115.0, 116.0, 117.75, 119.0, 118.75, 117.5, 117.75, 119.5, 120.0, 123.5, 125.0, 126.75, 126.25, 121.5, 117.75, 121.0, 121.0, 118.0, 116.0, 117.25, 119.0, 120.25, 119.25, 120.0, 120.25, 121